# Ejercicio 4


In [ ]:
import simpy
import random
import numpy as np

# Parámetros
np.random.seed(1)
random.seed(1)

LAMBDA = 1 / 5

MU = {
    1: 1 / 2,
    2: 1 / 4,
    3: 1 / 4,
    4: 1 / 3,
    5: 1 / 2
}

N_CLIENTES = 100000

# Crear entorno simpy
env = simpy.Environment()

chatbots = {
    i: simpy.Resource(env, capacity=1)
    for i in range(1, 6)
}

# Métricas
esperas = {i: [] for i in range(1, 6)}
tiempos_nodo = {i: [] for i in range(1, 6)}
ocupacion = {i: 0 for i in range(1, 6)}
visitas_nodo = {i: 0 for i in range(1, 6)}

tiempo_sistema = []


# Tránsito
def siguiente_nodo(actual):
    if actual == 1:
        return np.random.choice([2, 3], p=[0.7, 0.3])
    if actual == 2:
        return np.random.choice([3, 4], p=[0.6, 0.4])
    if actual == 3:
        return 4
    if actual == 4:
        return 5
    if actual == 5:
        return np.random.choice([1, 6], p=[0.1, 0.9])
    return 6


# Cliente
def cliente(env, nombre):
    llegada = env.now
    nodo = np.random.choice([1, 3, 4], p=[0.6, 0.1, 0.3])
    visitas = 0

    while nodo != 6:

        visitas += 1
        visitas_nodo[nodo] += 1

        recurso = chatbots[nodo]

        instante_cola = env.now

        with recurso.request() as req:

            yield req

            espera = env.now - instante_cola
            esperas[nodo].append(espera)

            servicio = random.expovariate(MU[nodo])

            yield env.timeout(servicio)

            fin_servicio = env.now

            tiempos_nodo[nodo].append(fin_servicio - instante_cola)
            ocupacion[nodo] += servicio

        nodo = siguiente_nodo(nodo)

    tiempo_sistema.append(env.now - llegada)

# Llegadas
def llegadas(env):

    for i in range(N_CLIENTES):

        env.process(cliente(env, f"C{i}"))

        interarrival = random.expovariate(LAMBDA)
        yield env.timeout(interarrival)

# Ejecutar
env.process(llegadas(env))
env.run()

# Resultados
print("\nTiempo medio de espera")

for i in range(1, 6):
    print(
        f"Chatbot {i}: "
        f"{np.mean(esperas[i]):.3f}"
    )

print("\nUtilización")

tiempo_total = env.now

for i in range(1, 6):

    rho = ocupacion[i] / tiempo_total

    print(
        f"Chatbot {i}: {rho:.3f}"
    )

print("\nNúmero medio en el sistema")


for i in range(1, 6):
    N = (visitas_nodo[i] / tiempo_total)* np.mean(tiempos_nodo[i])

    print(f"Chatbot {i}: {N:.3f}")

print("\nTiempo medio en sistema")

print(np.mean(tiempo_sistema))


Tiempo medio de espera
Chatbot 1: 0.779
Chatbot 2: 2.686
Chatbot 3: 3.755
Chatbot 4: 5.951
Chatbot 5: 1.626

Utilización
Chatbot 1: 0.285
Chatbot 2: 0.398
Chatbot 3: 0.487
Chatbot 4: 0.668
Chatbot 5: 0.446

Número medio en el sistema
Chatbot 1: 0.396
Chatbot 2: 0.666
Chatbot 3: 0.946
Chatbot 4: 1.992
Chatbot 5: 0.808

Tiempo medio en sistema
24.02261275273052
